# Cross-Model Lifecycle Desynchronization

**Companion experiment for** *The Knowledge Lifecycle of Large Language Models* (Thakur and Talele, 2026) · [Paper repository](https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE) · [Live demo](https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle)

**Environment:** Kaggle, GPU (T4 x2 or P100), internet on ·  **Runtime:** roughly 15 minutes


## 1. Problem

When a fact changes after a model's training data was collected, the weights hold the old version and a retrieved document holds the new one. The paper measures which one wins on GPT-2 base, using two numbers per probe:

- $\mathcal{D}_{sync} = -\ln P(y^* \mid \text{context} + \text{query})$, the surprisal of the correct answer with the corrective document present. Above 9.2 nats the correct answer is below $10^{-4}$ probability and no realistic decoding recovers it.
- $\mathcal{I}_{ctx} = D_{KL}(P_{ctx} \| P_{plain})$ over the full vocabulary, how far the document moves the model's distribution at all.

This notebook answers the question the paper poses in Section 9.2: **how do these numbers fall with model scale and instruction tuning?** Two axes are covered: scale within one family (GPT-2 base to large, same data and tokenizer, so scale is the only variable), and instruction tuning at small scale (chat-tuned models against the untuned GPT-2 family).

Everything is deterministic. There is no sampling anywhere, so identical inputs give identical numbers on every run, and no seed is required beyond what determinism already provides.


## 2. Setup

Kaggle images ship `torch` and `transformers` preinstalled; nothing is installed here so the run cannot drift with pip resolution.


In [ ]:
import gc
import math

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}, torch {torch.__version__}")

## 3. Probes

Three documented fact changes, byte-identical to the paper and the live demo so results are directly comparable. Each probe has a query that ends mid-sentence, a corrective context contradicting the model's parametric memory, and a single-word verified answer. On GPT-2 base fp32 these land in three regimes: resolution failure (vioxx, D_sync 12.05), the exposure trap (monarch, context boosts the wrong token), and drift under strong context (twitter, I_ctx 0.90 yet the top answer stays wrong).


In [ ]:
PROBES = [
    {
        "name": "vioxx",
        "query": "Question: Is Vioxx safe to prescribe? Answer: Vioxx is considered",
        "context": "Context: In September 2004, Merck voluntarily withdrew Vioxx after trials revealed increased cardiovascular risks.",
        "answer": " withdrawn",
    },
    {
        "name": "monarch",
        "query": "Question: Who is the current British monarch? Answer: The current British monarch is",
        "context": "Context: Queen Elizabeth II died in September 2022. Charles III acceded to the throne and is the reigning King of the United Kingdom.",
        "answer": " Charles",
    },
    {
        "name": "twitter",
        "query": "Question: What is the social network Twitter called today? Answer: Twitter is now called",
        "context": "Context: In July 2023, Twitter was rebranded as X under Elon Musk's ownership.",
        "answer": " X",
    },
]

MODELS = [
    "gpt2",                                  # 124M, the paper's reference model
    "gpt2-medium",                           # 355M  } same family, data and tokenizer,
    "gpt2-large",                            # 774M  } so scale is the only variable
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",    # chat-tuned at GPT-2-large scale
    "Qwen/Qwen2.5-0.5B-Instruct",            # instruction-tuned near GPT-2-medium scale
    "Qwen/Qwen2.5-1.5B-Instruct",            # largest that fits a T4 in fp16 comfortably
]

## 4. Measurement

One function per concept. The answer's **first token** is measured, exactly as the paper's estimator specifies; the tokenization is recorded per model because every tokenizer splits the answer differently, and a probe is only comparable across models with that piece on record. Logits are cast to fp32 before softmax so fp16 models are measured at full precision.


In [ ]:
def next_token_distribution(model, tokenizer, prompt):
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        logits = model(**inputs).logits[0, -1, :].float()
    return F.softmax(logits, dim=-1)


def run_probe(model, tokenizer, probe):
    p_plain = next_token_distribution(model, tokenizer, probe["query"])
    p_ctx = next_token_distribution(model, tokenizer, probe["context"] + "\n" + probe["query"])

    ids = tokenizer.encode(probe["answer"], add_special_tokens=False)
    first_piece = tokenizer.decode([ids[0]])
    pt_plain = max(p_plain[ids[0]].item(), 1e-12)   # floor keeps the log finite
    pt_ctx = max(p_ctx[ids[0]].item(), 1e-12)

    top_plain = tokenizer.decode([int(torch.argmax(p_plain))])
    top_ctx = tokenizer.decode([int(torch.argmax(p_ctx))])

    return {
        "probe": probe["name"],
        "answer_first_piece": repr(first_piece),
        "answer_pieces": len(ids),
        "P_answer_plain": pt_plain,
        "P_answer_ctx": pt_ctx,
        "D_sync_nats": -math.log(pt_ctx),
        "I_ctx_nats": F.kl_div(p_plain.log(), p_ctx, reduction="sum").item(),
        "top_token_plain": repr(top_plain),
        "top_token_ctx": repr(top_ctx),
        "correct_top1": top_ctx == first_piece,
    }

## 5. Run

Models are loaded one at a time and released before the next, because two fp16 models can exceed a single T4's memory. `torch_dtype="auto"` gives each model its native precision; measurement precision is handled at the logits (Section 4).


In [ ]:
rows = []
for name in MODELS:
    print(f"=== {name} ===")
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto", device_map=DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    for probe in PROBES:
        row = {"model": name, "params_M": round(n_params)}
        row.update(run_probe(model, tokenizer, probe))
        rows.append(row)
        print(f"  {row['probe']:<8} D_sync={row['D_sync_nats']:6.2f}  I_ctx={row['I_ctx_nats']:6.3f}  correct_top1={row['correct_top1']}")
    del model
    gc.collect()
    torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv("cross_model_dsync.csv", index=False)
results

## 6. Reproduction check

The gpt2/vioxx row must reproduce the paper's fp32 measurement before anything downstream is trusted: D_sync = 12.0464 nats, I_ctx = 0.0333 nats. The tolerance of 0.05 nats allows for library-version noise in the last digits while still catching any real deviation. **If this cell fails, stop and report; do not proceed to conclusions.**


In [ ]:
check = results[(results.model == "gpt2") & (results.probe == "vioxx")].iloc[0]
print(f"gpt2/vioxx: D_sync={check.D_sync_nats:.4f} (paper: 12.0464), I_ctx={check.I_ctx_nats:.4f} (paper: 0.0333)")
assert abs(check.D_sync_nats - 12.0464) < 0.05, "does not reproduce the paper's measurement"
print("reproduction check passed")

## 7. Results view

One pivot per metric, models as rows in size order, probes as columns. This is the table that goes back into the paper, so it is built from `results` rather than typed by hand.


In [ ]:
order = [m for m in MODELS]
for metric in ["D_sync_nats", "I_ctx_nats", "correct_top1"]:
    pivot = results.pivot(index="model", columns="probe", values=metric).reindex(order)
    print(f"--- {metric} ---")
    print(pivot.round(3).to_string())
    print()

## 8. Findings

*Written after the run, from the tables above. One short paragraph per question; state the numbers that support each sentence. Nothing that failed is deleted: if a model behaves strangely, that is a finding.*

1. **Scale within a family.** Does D_sync fall from gpt2 to gpt2-large on each probe, and by how much?
2. **Instruction tuning.** Do the chat-tuned models resolve the conflicts (`correct_top1` true) where similar-scale base models do not?
3. **The exposure trap.** On the monarch probe, does any model repeat the paper's pattern where the corrective context increases the probability of the wrong continuation? Compare `P_answer_plain` against the probability of the stale token in both conditions.

*Attach `cross_model_dsync.csv` alongside the written findings.*
